# Dataset Loading

In [1]:
import json
import pandas as pd

database = "../data/raw_credit_applications.json"

with open(database, "r", encoding="utf-8") as f:
    raw = json.load(f)

type(raw), len(raw)


(list, 502)

In [2]:
print(raw[0].keys()) # checking what fields exist at the top level

print(raw[0]) # checking the first record of the database fully

dict_keys(['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision', 'processing_timestamp'])
{'_id': 'app_200', 'applicant_info': {'full_name': 'Jerry Smith', 'email': 'jerry.smith17@hotmail.com', 'ssn': '596-64-4340', 'ip_address': '192.168.48.155', 'gender': 'Male', 'date_of_birth': '2001-03-09', 'zip_code': '10036'}, 'financials': {'annual_income': 73000, 'credit_history_months': 23, 'debt_to_income': 0.2, 'savings_balance': 31212}, 'spending_behavior': [{'category': 'Shopping', 'amount': 480}, {'category': 'Rent', 'amount': 790}, {'category': 'Alcohol', 'amount': 247}], 'decision': {'loan_approved': False, 'rejection_reason': 'algorithm_risk_score'}, 'processing_timestamp': '2024-01-15T00:00:00Z'}


In [3]:
df = pd.json_normalize(raw, sep=".") # converting the nested JSON into a pandas dataframe


df.head(3)

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,...,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,...,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,...,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN


By taking a first look at the dataset we can see that the spending_behaviour column is different from the other ones. It is a list of dictionaries so we need to adress that. What I will do now is transform the list of dictionaries into seperate columns


In [4]:
# Step 1: explode list
sp = df[["_id", "spending_behavior"]].explode("spending_behavior")

# Step 2: normalize dictionaries
sp_norm = pd.json_normalize(sp["spending_behavior"])
sp_norm.index = sp.index

sp = pd.concat([sp[["_id"]], sp_norm], axis=1)

# Step 3: ensure numeric
sp["amount"] = pd.to_numeric(sp["amount"], errors="coerce")

# Step 4: pivot to wide format
sp_pivot = (
    sp.pivot_table(
        index="_id",
        columns="category",
        values="amount",
        aggfunc="sum",
        fill_value=0
    )
    .add_prefix("spend_")
    .reset_index()
)

# Step 5: merge back to main df
df = df.drop(columns=["spending_behavior"]).merge(sp_pivot, on="_id", how="left")

In [5]:
df.head(3)

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,...,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,...,0,0,0,0,0,790,480,0,0,0
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,...,0,0,0,243,0,608,0,0,0,0
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,...,0,0,0,0,0,109,0,0,0,0


In [6]:
df.columns

Index(['_id', 'processing_timestamp', 'applicant_info.full_name',
       'applicant_info.email', 'applicant_info.ssn',
       'applicant_info.ip_address', 'applicant_info.gender',
       'applicant_info.date_of_birth', 'applicant_info.zip_code',
       'financials.annual_income', 'financials.credit_history_months',
       'financials.debt_to_income', 'financials.savings_balance',
       'decision.loan_approved', 'decision.rejection_reason', 'loan_purpose',
       'decision.interest_rate', 'decision.approved_amount',
       'financials.annual_salary', 'notes', 'spend_Adult Entertainment',
       'spend_Alcohol', 'spend_Dining', 'spend_Education',
       'spend_Entertainment', 'spend_Fitness', 'spend_Gambling',
       'spend_Groceries', 'spend_Healthcare', 'spend_Insurance', 'spend_Rent',
       'spend_Shopping', 'spend_Transportation', 'spend_Travel',
       'spend_Utilities'],
      dtype='object')

# Missing Values Analysis



We will now conduct the missing values analysis. As a missing value we mean NaN or blank text

In [7]:
import numpy as np

# Replace empty or whitespace-only strings with NaN
df = df.replace(r'^\s*$', np.nan, regex=True)

missing_counts = df.isna().sum()
missing_percent = (df.isna().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent
}).sort_values("missing_count", ascending=False)

missing_summary

,missing_count,missing_percent
notes,500,99.60
financials.annual_salary,497,99.00
loan_purpose,452,90.04
processing_timestamp,440,87.65
decision.rejection_reason,292,58.17
decision.approved_amount,210,41.83
decision.interest_rate,210,41.83
applicant_info.email,7,1.39
applicant_info.date_of_birth,5,1.00
financials.annual_income,5,1.00


The decision.rejection_reason varible has a 58% miissing rate which is quite high. However this is a variable that should only have text if the loan_approved = False. So we will now check if there are cases where that is not the case. If the output of the following code is 0 then this means this part of the dataset is consistent.


In [8]:
# rejection_reason should exist only if loan_approved == False
invalid_rejection = df[
    (df["decision.loan_approved"] == True) &
    (df["decision.rejection_reason"].notna())
]

len(invalid_rejection)

0

Checking if there are cases where the loan was not approved without any justification. If the ouput of the code is 0 it means that all rejected loans have a justification

In [9]:
invalid_missing_reason = df[
    (df["decision.loan_approved"] == False) &
    (df["decision.rejection_reason"].isna())
]

len(invalid_missing_reason)

0

Checking if there are cases where the loan was not approved but amount or interest columns have values which should not happen

In [10]:
invalid_approved_fields = df[
    (df["decision.loan_approved"] == False) &
    (
        df["decision.approved_amount"].notna() |
        df["decision.interest_rate"].notna()
    )
]

len(invalid_approved_fields)

0

Checking if there are cases where the loan was approved but there are missing values in the amount or interest column

In [11]:
invalid_missing_approved_fields = df[
    (df["decision.loan_approved"] == True) &
    (
        df["decision.approved_amount"].isna() |
        df["decision.interest_rate"].isna()
    )
]

len(invalid_missing_approved_fields)

0

# Duplicate Values Analysis


Total number of rows before any filtering:

In [12]:
rows_before = df.shape[0]
print("Rows before duplicate removal:", rows_before)

Rows before duplicate removal: 502


In [13]:
df["_id"].duplicated().sum()

2

These are the rows that have the same application id. 

In [14]:
pd.set_option("display.max_columns", None) # code to show all columns

By checking the Notes column we can see that for one of the duplicated entries the system detected a duplicate submission and for the other the applicant resubmitted the application.

In [15]:
df[df["_id"].duplicated(keep=False)].sort_values("_id") 


,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities
383,app_001,NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,427-90-1892,10.121.120.213,Female,1986-05-27,90230,102000,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,1152,0,0,0,0,0,0,0,0,0
455,app_001,NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,NaN,NaN,NaN,NaN,NaN,102000,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,NaN,DUPLICATE_ENTRY_ERROR,0,0,0,0,0,1152,0,0,0,0,0,0,0,0,0
8,app_042,NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,69000,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,936,0,0,0,0,0,0,306,0,0,0,0,0
354,app_042,NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,69000,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,RESUBMISSION,0,0,936,0,0,0,0,0,0,306,0,0,0,0,0


For app_001 duplicate. I removed 455 because it less complete with information than row 383.

For app_042 duplicate, the rows have the exact same information except for the 'Notes' column so I removed row 354 which has the Note saying it was a 'Resubmition'.

In [16]:
df = df.drop(index=[455, 354])

df.loc[[8, 383]]



,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities
8,app_042,NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,69000,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,936,0,0,0,0,0,0,306,0,0,0,0,0
383,app_001,NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,427-90-1892,10.121.120.213,Female,1986-05-27,90230,102000,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,1152,0,0,0,0,0,0,0,0,0


If there is the same SSN more than 1 time this means that the same person applied more than once.

In [17]:
df["applicant_info.ssn"].duplicated().sum()

5

In [18]:
ssn_col = "applicant_info.ssn"

# 1) SSN duplicates excluding NaN SSNs
df_ssn = df[df[ssn_col].notna()].copy()
n_dup_ssn_extra = df_ssn[ssn_col].duplicated().sum()
n_dup_ssn_rows = df_ssn[df_ssn[ssn_col].duplicated(keep=False)].shape[0]


# 2) Show the duplicate SSN groups with key identity fields
df_flagged = (
    df_ssn[df_ssn[ssn_col].duplicated(keep=False)]
    .sort_values(ssn_col)[["_id","applicant_info.full_name","applicant_info.gender","applicant_info.date_of_birth",ssn_col]]
)
df_flagged

,_id,applicant_info.full_name,applicant_info.gender,applicant_info.date_of_birth,applicant_info.ssn
92,app_088,Susan Martinez,F,1986-10-15,780-24-9300
122,app_016,Gary Wilson,M,1959-12-11,780-24-9300
16,app_101,Sandra Smith,Female,1997-03-23,937-72-8731
499,app_234,Samuel Hill,Male,1976/01/29,937-72-8731


There are 2 cases here: 
- Case 1: SSN: 780-24-9300

app_088 → Susan Martinez (F, 1986-10-15)

app_016 → Gary Wilson (M, 1959-12-11)

- Case 2: SSN: 937-72-8731

app_101 → Sandra Smith (Female, 1997-03-23)

app_234 → Samuel Hill (Male, 1976/01/29)







In [19]:
ssn_col = "applicant_info.ssn"

df["ssn_collision"] = (
    df[df[ssn_col].notna()]
      .groupby(ssn_col)["applicant_info.full_name"]
      .transform("nunique")
      .reindex(df.index, fill_value=0)
      .gt(1)
)

We identified  cases where the same SSN was associated with different applicant identities (name, gender, or date of birth).

Since SSN should uniquely identify an individual, this represents a serious data validity issue.

To preserve data integrity while maintaining traceability, we retain the original dataset and create a cleaned analytical version excluding these flagged records.

In [20]:
df_clean = df[~df["ssn_collision"]].copy()

In [21]:
df_clean.head()

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities,ssn_collision
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,247,0,0,0,0,0,0,0,0,790,480,0,0,0,False
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,96,0,0,0,0,0,243,0,608,0,0,0,0,False
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,109,0,0,0,0,False
3,app_024,NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN,0,0,0,0,0,575,0,0,0,0,0,0,0,0,0,False
4,app_184,2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,57000,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,463,0,0,0,0,0,0,0,0,0,0,False


After excluding missing values, no duplicate email addresses were identified. The apparent duplicates were due to multiple missing email entries, which represent completeness issues rather than identity collisions.

In [22]:
df[df["applicant_info.email"].notna()]["applicant_info.email"].duplicated().sum()

0

Checking if there are entries with the same Full Name + Date of Birth:

In [23]:
df.duplicated(  
    subset=["applicant_info.full_name", "applicant_info.date_of_birth"]
).sum()

0

### Up to this point in total 6 rows were removed from the original dataset (2 of them were because they were true duplicates of another row), the other 4 were due to a conflict regarding SSN´s. The current cleaned dataset df_clean has 496 rows while the original one has 502

In [24]:
len(df_clean)

496

# Standardize Data Types

In [25]:
df_clean.head(20)

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities,ssn_collision
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,247,0,0,0,0,0,0,0,0,790,480,0,0,0,False
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,96,0,0,0,0,0,243,0,608,0,0,0,0,False
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,109,0,0,0,0,False
3,app_024,NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN,0,0,0,0,0,575,0,0,0,0,0,0,0,0,0,False
4,app_184,2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,57000,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,463,0,0,0,0,0,0,0,0,0,0,False
5,app_275,NaN,Maria Miller,maria.miller67@outlook.com,417-25-4912,172.25.58.70,F,14/02/1982,10019,110000,33,0.05,49933,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,571,0,0,0,0,0,0,0,0,0,0,False
6,app_099,NaN,Nicholas King,nicholas.king46@outlook.com,613-23-2503,10.62.62.45,Male,28/01/1990,10022,55000,61,0.17,30159,True,NaN,NaN,5.6,27000.0,NaN,NaN,0,0,458,0,0,0,0,0,0,0,0,0,0,0,0,False
7,app_246,NaN,Susan Rivera,susan.rivera74@gmail.com,176-97-1864,192.168.158.59,F,1991-10-11,90223,82000,31,0.29,21809,True,NaN,auto,2.8,38000.0,NaN,NaN,0,0,0,0,0,0,0,0,478,0,0,0,0,0,0,False
8,app_042,NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,69000,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,936,0,0,0,0,0,0,306,0,0,0,0,0,False
9,app_348,NaN,Michael Mitchell,michael.mitchell42@hotmail.com,100-94-8400,172.28.12.121,Male,1989-10-10,10080,55000,5,0.41,13794,False,insufficient_credit_history,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,199,0,0,0,0,0,0,0,851,0,False


Standardize gender (Male/Female/Unknown)

In [26]:
# 1) Normalize gender values into a consistent set: Male, Female, Unknown
gender_map = {
    "M": "Male",
    "MALE": "Male",
    "F": "Female",
    "FEMALE": "Female"
}

df_clean["applicant_info.gender"] = (
    df_clean["applicant_info.gender"]
      .astype("string")
      .str.strip()
      .str.upper()
      .map(gender_map)  # maps M/F/MALE/FEMALE -> Male/Female
      .fillna(df_clean["applicant_info.gender"].astype("string").str.strip())  # keep already-good values
)

df_clean["applicant_info.gender"] = df_clean["applicant_info.gender"].replace({"": pd.NA}).fillna("Unknown")

In [27]:
# Standardize DOB from multiple formats into a single datetime column (NaT if truly invalid)

s = (
    df_clean["applicant_info.date_of_birth"]
      .astype("string")
      .str.strip()
      .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
)

# 1) Unambiguous formats first
d_ymd_dash  = pd.to_datetime(s, errors="coerce", format="%Y-%m-%d")  # 2001-03-09
d_ymd_slash = pd.to_datetime(s, errors="coerce", format="%Y/%m/%d")  # 1990/07/26

# 2) Slash ambiguous: try both
d_dmy_slash = pd.to_datetime(s, errors="coerce", format="%d/%m/%Y")  # 14/02/1982
d_mdy_slash = pd.to_datetime(s, errors="coerce", format="%m/%d/%Y")  # 03/20/1968

# 3) Dash ambiguous: try both (then choose using >12 rule)
mask_dash = s.str.match(r"^\d{2}-\d{2}-\d{4}$", na=False)
part1 = pd.to_numeric(s.str.split("-", expand=True)[0], errors="coerce")  # first chunk

d_dmy_dash = pd.to_datetime(s.where(mask_dash), errors="coerce", format="%d-%m-%Y")
d_mdy_dash = pd.to_datetime(s.where(mask_dash), errors="coerce", format="%m-%d-%Y")
d_dash = d_dmy_dash.where(part1 > 12, d_mdy_dash)

# 4) Combine (priority order)
df_clean["applicant_info.date_of_birth"] = (
    d_ymd_dash
      .fillna(d_ymd_slash)
      .fillna(d_dmy_slash)
      .fillna(d_mdy_slash)
      .fillna(d_dash)
)

In [28]:
df_clean.head(30)

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities,ssn_collision
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,247,0,0,0,0,0,0,0,0,790,480,0,0,0,False
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,96,0,0,0,0,0,243,0,608,0,0,0,0,False
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,109,0,0,0,0,False
3,app_024,NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN,0,0,0,0,0,575,0,0,0,0,0,0,0,0,0,False
4,app_184,2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080,57000,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,463,0,0,0,0,0,0,0,0,0,0,False
5,app_275,NaN,Maria Miller,maria.miller67@outlook.com,417-25-4912,172.25.58.70,Female,1982-02-14,10019,110000,33,0.05,49933,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,571,0,0,0,0,0,0,0,0,0,0,False
6,app_099,NaN,Nicholas King,nicholas.king46@outlook.com,613-23-2503,10.62.62.45,Male,1990-01-28,10022,55000,61,0.17,30159,True,NaN,NaN,5.6,27000.0,NaN,NaN,0,0,458,0,0,0,0,0,0,0,0,0,0,0,0,False
7,app_246,NaN,Susan Rivera,susan.rivera74@gmail.com,176-97-1864,192.168.158.59,Female,1991-10-11,90223,82000,31,0.29,21809,True,NaN,auto,2.8,38000.0,NaN,NaN,0,0,0,0,0,0,0,0,478,0,0,0,0,0,0,False
8,app_042,NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,69000,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,936,0,0,0,0,0,0,306,0,0,0,0,0,False
9,app_348,NaN,Michael Mitchell,michael.mitchell42@hotmail.com,100-94-8400,172.28.12.121,Male,1989-10-10,10080,55000,5,0.41,13794,False,insufficient_credit_history,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,199,0,0,0,0,0,0,0,851,0,False


Zip Codes should be considered strings:

In [29]:
df_clean["applicant_info.zip_code"].map(type).value_counts()

applicant_info.zip_code
<class 'str'>      495
<class 'float'>      1
Name: count, dtype: int64

In [30]:
df_clean.loc[
    df_clean["applicant_info.zip_code"].map(type) == float,
    ["_id", "applicant_info.zip_code"]
]

,_id,applicant_info.zip_code
26,app_075,NaN


We standardize all zip code records into panda strings, which automatically makes the NaN record a pd.NaN

In [31]:
df_clean["applicant_info.zip_code"] = (
    df_clean["applicant_info.zip_code"]
        .astype("string")
        .str.strip()
)

Standardize processing_timestamp into datetime:

In [32]:
df_clean["processing_timestamp"] = pd.to_datetime(
    df_clean["processing_timestamp"],
    errors="coerce",
    utc=True
)

Standardizing annual_income to numeric:

In [33]:
df_clean["financials.annual_income"] = pd.to_numeric(
    df_clean["financials.annual_income"],
    errors="coerce"
)

In [34]:
text_cols = [
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "loan_purpose",
    "notes",
    "decision.rejection_reason"
]

for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = (
            df_clean[col]
                .astype("string")
                .str.strip()
                .replace({"": pd.NA})
        )

In [35]:
df_clean[text_cols].dtypes

applicant_info.full_name     string[python]
applicant_info.email         string[python]
applicant_info.ssn           string[python]
applicant_info.ip_address    string[python]
loan_purpose                 string[python]
notes                        string[python]
decision.rejection_reason    string[python]
dtype: object

In [36]:
df_clean.dtypes

_id                                              object
processing_timestamp                datetime64[ns, UTC]
applicant_info.full_name                 string[python]
applicant_info.email                     string[python]
applicant_info.ssn                       string[python]
applicant_info.ip_address                string[python]
applicant_info.gender                            object
applicant_info.date_of_birth             datetime64[ns]
applicant_info.zip_code                  string[python]
financials.annual_income                        float64
financials.credit_history_months                  int64
financials.debt_to_income                       float64
financials.savings_balance                        int64
decision.loan_approved                             bool
decision.rejection_reason                string[python]
loan_purpose                             string[python]
decision.interest_rate                          float64
decision.approved_amount                        

# Data Validity and Cohesion

Income and savings should never be negative.

In [37]:
df_clean.loc[
    (df_clean["financials.annual_income"] < 0) |
    (df_clean["financials.savings_balance"] < 0),
    ["_id", "financials.annual_income", "financials.savings_balance"]
]

,_id,financials.annual_income,financials.savings_balance
159,app_290,104000.0,-5000


Remove app_290 from df_clean

In [38]:
row_to_flag = df_clean[df_clean["_id"] == "app_290"].copy()
df_clean = df_clean[df_clean["_id"] != "app_290"]

Add it to df_flagged

In [39]:
row_to_flag["flag_reason"] = "Negative savings balance"
df_flagged = pd.concat([df_flagged, row_to_flag])

Checking whether there are applicants that have spendings total greater than their income:

In [40]:
spend_cols = [c for c in df_clean.columns if c.startswith("spend_")]
total_spend = df_clean[spend_cols].sum(axis=1)

df_clean.loc[
    total_spend > df_clean["financials.annual_income"],
    ["_id", "financials.annual_income"]
]

,_id,financials.annual_income
426,app_190,0.0


Extract and remove app_190 from df_clean

In [41]:
row_to_flag = df_clean[df_clean["_id"] == "app_190"].copy()
df_clean = df_clean[df_clean["_id"] != "app_190"]

Add it to df_flagged

In [42]:
row_to_flag["flag_reason"] = "Zero income but positive spending"
df_flagged = pd.concat([df_flagged, row_to_flag], ignore_index=False)

In [43]:
df_clean.head()

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities,ssn_collision
0,app_200,2024-01-15 00:00:00+00:00,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000.0,23,0.20,31212,False,algorithm_risk_score,<NA>,NaN,NaN,NaN,<NA>,0,247,0,0,0,0,0,0,0,0,790,480,0,0,0,False
1,app_037,NaT,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032,78000.0,51,0.18,17915,False,algorithm_risk_score,<NA>,NaN,NaN,NaN,<NA>,0,0,96,0,0,0,0,0,243,0,608,0,0,0,0,False
2,app_215,NaT,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000.0,41,0.21,37909,True,<NA>,vacation,3.7,59000.0,NaN,<NA>,0,0,0,0,0,0,0,0,0,0,109,0,0,0,0,False
3,app_024,NaT,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000.0,70,0.35,0,True,<NA>,<NA>,4.3,34000.0,NaN,<NA>,0,0,0,0,0,575,0,0,0,0,0,0,0,0,0,False
4,app_184,2024-01-15 00:00:00+00:00,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080,57000.0,14,0.23,31763,False,algorithm_risk_score,<NA>,NaN,NaN,NaN,<NA>,0,0,0,0,463,0,0,0,0,0,0,0,0,0,0,False


Created an age column to check for impossible ages regarding credit decicions, must be over 18 and cant be over 100 years old:

In [44]:
df_clean["age"] = (
    (pd.Timestamp.today() - df_clean["applicant_info.date_of_birth"])
    .dt.days // 365
)

In [45]:
df_clean[
    (df_clean["age"] < 18) |
    (df_clean["age"] > 100)
]

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities,ssn_collision,age


Checking whether an applicant has had credit longer than it is legally possible:

In [46]:
df_clean[
    (df_clean["financials.credit_history_months"] / 12) >
    (df_clean["age"] - 18)
][["_id", "age", "financials.credit_history_months"]]

,_id,age,financials.credit_history_months
496,app_049,25.0,92


In [47]:
row_to_flag = df_clean[df_clean["_id"] == "app_049"].copy()
row_to_flag["flag_reason"] = "Credit history exceeds legal possible limit"
df_flagged = pd.concat([df_flagged, row_to_flag])

In [48]:
df_clean = df_clean[df_clean["_id"] != "app_049"]

Checking whether credit history months is >= 0 :

In [49]:
df_clean[
    (df_clean["financials.credit_history_months"] < 0) |
    (df_clean["financials.credit_history_months"] % 1 != 0)
][["_id","financials.credit_history_months"]]

,_id,financials.credit_history_months
137,app_043,-10
162,app_156,-3


In [50]:
# Extract invalid rows
rows_to_flag = df_clean[
    (df_clean["financials.credit_history_months"] < 0) |
    (df_clean["financials.credit_history_months"] % 1 != 0)
].copy()

# Add flag reason
rows_to_flag["flag_reason"] = "Invalid credit history (negative)"

# Append to df_flagged
df_flagged = pd.concat([df_flagged, rows_to_flag], ignore_index=True)

# Remove from df_clean
df_clean = df_clean.drop(rows_to_flag.index)

Checking for impossible Debt to Income Ratio values (should be 0<=DTI<=1). We have 1 case where this happens (dont know what to do, for now im gonna consider it is possible)

In [51]:
df_clean[
    (df_clean["financials.debt_to_income"] < 0) |
    (df_clean["financials.debt_to_income"] > 1)
]

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities,ssn_collision,age
316,app_402,NaT,Heather Flores,heather.flores23@mail.com,100-58-8097,172.21.176.42,Female,1965-03-07,90214,88000.0,27,1.85,37281,True,<NA>,<NA>,3.2,17000.0,NaN,<NA>,0,0,0,482,0,0,0,0,0,0,0,0,0,0,0,False,61.0


# Final Adjustment of Columns and Records in df_clean

In this part I will just some final adjustments to our two datasets df_clean which is the one we will use for our analysis and df_flagged, a dataset with all rows removed because where there was no note explaining what the issue with record was.  The flagged dataset containt a columns called flag_reason which explain why they were removed from the dataset 

In [52]:
df_clean.columns


Index(['_id', 'processing_timestamp', 'applicant_info.full_name',
       'applicant_info.email', 'applicant_info.ssn',
       'applicant_info.ip_address', 'applicant_info.gender',
       'applicant_info.date_of_birth', 'applicant_info.zip_code',
       'financials.annual_income', 'financials.credit_history_months',
       'financials.debt_to_income', 'financials.savings_balance',
       'decision.loan_approved', 'decision.rejection_reason', 'loan_purpose',
       'decision.interest_rate', 'decision.approved_amount',
       'financials.annual_salary', 'notes', 'spend_Adult Entertainment',
       'spend_Alcohol', 'spend_Dining', 'spend_Education',
       'spend_Entertainment', 'spend_Fitness', 'spend_Gambling',
       'spend_Groceries', 'spend_Healthcare', 'spend_Insurance', 'spend_Rent',
       'spend_Shopping', 'spend_Transportation', 'spend_Travel',
       'spend_Utilities', 'ssn_collision', 'age'],
      dtype='object')

In [53]:
df_clean.head()

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities,ssn_collision,age
0,app_200,2024-01-15 00:00:00+00:00,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000.0,23,0.20,31212,False,algorithm_risk_score,<NA>,NaN,NaN,NaN,<NA>,0,247,0,0,0,0,0,0,0,0,790,480,0,0,0,False,24.0
1,app_037,NaT,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032,78000.0,51,0.18,17915,False,algorithm_risk_score,<NA>,NaN,NaN,NaN,<NA>,0,0,96,0,0,0,0,0,243,0,608,0,0,0,0,False,33.0
2,app_215,NaT,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000.0,41,0.21,37909,True,<NA>,vacation,3.7,59000.0,NaN,<NA>,0,0,0,0,0,0,0,0,0,0,109,0,0,0,0,False,36.0
3,app_024,NaT,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000.0,70,0.35,0,True,<NA>,<NA>,4.3,34000.0,NaN,<NA>,0,0,0,0,0,575,0,0,0,0,0,0,0,0,0,False,42.0
4,app_184,2024-01-15 00:00:00+00:00,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080,57000.0,14,0.23,31763,False,algorithm_risk_score,<NA>,NaN,NaN,NaN,<NA>,0,0,0,0,463,0,0,0,0,0,0,0,0,0,0,False,26.0


Just adding a justification to flag_reason for the record which had duplicate SSN´s

In [54]:
df_flagged.loc[df_flagged.index[:4], "flag_reason"] = "Duplicate SSNs"

Remove finnacial.annual salary column it was created at the beggining when loading the json dataset but it is the same as financial.annual_income but with 99% missing values

In [55]:
df_clean = df_clean.drop(columns=["financials.annual_salary"], errors="ignore")

In [56]:
df_clean.head()

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities,ssn_collision,age
0,app_200,2024-01-15 00:00:00+00:00,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000.0,23,0.20,31212,False,algorithm_risk_score,<NA>,NaN,NaN,<NA>,0,247,0,0,0,0,0,0,0,0,790,480,0,0,0,False,24.0
1,app_037,NaT,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032,78000.0,51,0.18,17915,False,algorithm_risk_score,<NA>,NaN,NaN,<NA>,0,0,96,0,0,0,0,0,243,0,608,0,0,0,0,False,33.0
2,app_215,NaT,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000.0,41,0.21,37909,True,<NA>,vacation,3.7,59000.0,<NA>,0,0,0,0,0,0,0,0,0,0,109,0,0,0,0,False,36.0
3,app_024,NaT,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000.0,70,0.35,0,True,<NA>,<NA>,4.3,34000.0,<NA>,0,0,0,0,0,575,0,0,0,0,0,0,0,0,0,False,42.0
4,app_184,2024-01-15 00:00:00+00:00,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080,57000.0,14,0.23,31763,False,algorithm_risk_score,<NA>,NaN,NaN,<NA>,0,0,0,0,463,0,0,0,0,0,0,0,0,0,0,False,26.0


remove ssn_collision was created for jsut for the removal of the duplicate ssn records

In [57]:
df_clean = df_clean.drop(columns=["ssn_collision"], errors="ignore")

In [58]:
df_clean.columns

Index(['_id', 'processing_timestamp', 'applicant_info.full_name',
       'applicant_info.email', 'applicant_info.ssn',
       'applicant_info.ip_address', 'applicant_info.gender',
       'applicant_info.date_of_birth', 'applicant_info.zip_code',
       'financials.annual_income', 'financials.credit_history_months',
       'financials.debt_to_income', 'financials.savings_balance',
       'decision.loan_approved', 'decision.rejection_reason', 'loan_purpose',
       'decision.interest_rate', 'decision.approved_amount', 'notes',
       'spend_Adult Entertainment', 'spend_Alcohol', 'spend_Dining',
       'spend_Education', 'spend_Entertainment', 'spend_Fitness',
       'spend_Gambling', 'spend_Groceries', 'spend_Healthcare',
       'spend_Insurance', 'spend_Rent', 'spend_Shopping',
       'spend_Transportation', 'spend_Travel', 'spend_Utilities', 'age'],
      dtype='object')

Renaming the columns into a more user friendly name:

In [59]:
df_clean.columns = (
    df_clean.columns
        .str.replace("applicant_info.", "", regex=False)
        .str.replace("financials.", "", regex=False)
        .str.replace("decision.", "", regex=False)
        .str.replace(".", "_", regex=False)
        .str.lower()
)

Changing "_id" to "application_id"

In [60]:
df_clean = df_clean.rename(columns={"_id": "application_id"})

Renaming the collumns in df_flagged the same as the ones in df_clean:

In [61]:
df_flagged.columns = (
    df_flagged.columns
        .str.replace("applicant_info.", "", regex=False)
        .str.replace("financials.", "", regex=False)
        .str.replace("decision.", "", regex=False)
        .str.replace(".", "_", regex=False)
        .str.lower()
)

df_flagged = df_flagged.rename(columns={"_id": "application_id"})

In [62]:
df_flagged


,application_id,full_name,gender,date_of_birth,ssn,processing_timestamp,email,ip_address,zip_code,annual_income,credit_history_months,debt_to_income,savings_balance,loan_approved,rejection_reason,loan_purpose,interest_rate,approved_amount,annual_salary,notes,spend_adult entertainment,spend_alcohol,spend_dining,spend_education,spend_entertainment,spend_fitness,spend_gambling,spend_groceries,spend_healthcare,spend_insurance,spend_rent,spend_shopping,spend_transportation,spend_travel,spend_utilities,ssn_collision,flag_reason,age
0,app_088,Susan Martinez,F,1986-10-15,780-24-9300,NaT,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Duplicate SSNs,NaN
1,app_016,Gary Wilson,M,1959-12-11,780-24-9300,NaT,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Duplicate SSNs,NaN
2,app_101,Sandra Smith,Female,1997-03-23,937-72-8731,NaT,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Duplicate SSNs,NaN
3,app_234,Samuel Hill,Male,1976/01/29,937-72-8731,NaT,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Duplicate SSNs,NaN
4,app_290,Stephanie Perez,Female,1990-01-06 00:00:00,866-48-3726,NaT,stephanie.perez69@gmail.com,172.25.130.219,90221,104000.0,39.0,0.27,-5000.0,True,<NA>,<NA>,4.3,49000.0,NaN,<NA>,0.0,0.0,0.0,0.0,0.0,599.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,Negative savings balance,NaN
5,app_190,Heather White,Female,1999-10-05 00:00:00,217-44-8191,2024-01-15 00:00:00+00:00,heather.white42@icloud.com,192.168.242.9,90297,0.0,0.0,0.14,19603.0,False,algorithm_risk_score,<NA>,NaN,NaN,NaN,<NA>,0.0,0.0,317.0,0.0,0.0,849.0,0.0,0.0,0.0,0.0,0.0,223.0,0.0,0.0,0.0,False,Zero income but positive spending,NaN
6,app_049,Donna Gonzalez,Female,2000-05-22 00:00:00,845-50-3562,NaT,donna.gonzalez12@yahoo.com,192.168.14.85,90278,59000.0,92.0,0.36,31881.0,False,algorithm_risk_score,<NA>,NaN,NaN,NaN,<NA>,0.0,0.0,0.0,0.0,848.0,444.0,0.0,131.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,Credit history exceeds legal possible limit,25.0
7,app_043,Daniel King,Male,1985-08-17 00:00:00,166-23-2000,NaT,daniel.king27@mail.com,10.204.248.60,10038,131000.0,-10.0,0.06,53098.0,True,<NA>,<NA>,3.9,68000.0,NaN,<NA>,0.0,0.0,0.0,0.0,464.0,0.0,0.0,398.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,Invalid credit history (negative),40.0
8,app_156,Jessica Green,Female,1999-11-28 00:00:00,801-19-4595,2024-01-15 00:00:00+00:00,jessica.green86@gmail.com,192.168.59.201,90239,25000.0,-3.0,0.21,13641.0,False,algorithm_risk_score,<NA>,NaN,NaN,NaN,<NA>,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,218.0,0.0,0.0,0.0,0.0,False,Invalid credit history (negative),26.0


In [63]:
len(df_clean)

491

In [64]:
df_clean.isna().sum().sort_values(ascending=False)

notes                        491
loan_purpose                 442
processing_timestamp         431
rejection_reason             290
approved_amount              201
interest_rate                201
email                          7
annual_income                  5
age                            4
date_of_birth                  4
ip_address                     4
ssn                            4
zip_code                       1
spend_fitness                  0
spend_utilities                0
spend_travel                   0
spend_transportation           0
spend_shopping                 0
spend_rent                     0
spend_insurance                0
spend_healthcare               0
spend_groceries                0
spend_gambling                 0
spend_education                0
spend_entertainment            0
credit_history_months          0
spend_dining                   0
spend_alcohol                  0
spend_adult entertainment      0
full_name                      0
gender    

Checking rows in df_clean that are missing either annual income, age, email or ssn:

In [65]:
missing_any = df_clean[
    df_clean["annual_income"].isna() |
    df_clean["age"].isna() |
    df_clean["email"].isna() |
    df_clean["ssn"].isna()
]

missing_any[["application_id", "annual_income", "age", "email", "ssn"]]

,application_id,annual_income,age,email,ssn
26,app_075,61000.0,NaN,<NA>,<NA>
76,app_436,NaN,27.0,amanda.torres59@gmail.com,135-51-1195
94,app_421,NaN,43.0,donald.baker60@icloud.com,344-50-4287
99,app_479,NaN,42.0,sandra.jones59@outlook.com,424-34-1670
141,app_463,NaN,50.0,emma.clark61@outlook.com,976-47-3536
149,app_449,NaN,57.0,lisa.roberts99@outlook.com,833-71-7935
187,app_413,92000.0,39.0,<NA>,584-37-2562
275,app_120,103000.0,NaN,<NA>,<NA>
297,app_268,112000.0,54.0,<NA>,<NA>
298,app_377,116000.0,35.0,<NA>,617-17-3415


In [66]:
# Identify rows missing age or annual income
rows_missing_core = df_clean[
    df_clean["age"].isna() |
    df_clean["annual_income"].isna()
].copy()

# Remove them from df_clean
df_clean = df_clean.drop(rows_missing_core.index).reset_index(drop=True)

In [67]:
# Add flag reason
rows_missing_core["flag_reason"] = "Missing either age or annual income"

# Append to df_flagged
df_flagged = pd.concat([df_flagged, rows_missing_core], ignore_index=True)

Records with missing annual_income or age were removed, as these are core variables required for credit risk analysis and fairness evaluation. Applications missing only non-essential PII fields such as email or ssn were retained, since these attributes are not required for modeling.

This ensures the final dataset is structurally valid and analysis-ready while preserving as many usable records as possible.

In [68]:
df_flagged

,application_id,full_name,gender,date_of_birth,ssn,processing_timestamp,email,ip_address,zip_code,annual_income,credit_history_months,debt_to_income,savings_balance,loan_approved,rejection_reason,loan_purpose,interest_rate,approved_amount,annual_salary,notes,spend_adult entertainment,spend_alcohol,spend_dining,spend_education,spend_entertainment,spend_fitness,spend_gambling,spend_groceries,spend_healthcare,spend_insurance,spend_rent,spend_shopping,spend_transportation,spend_travel,spend_utilities,ssn_collision,flag_reason,age
0,app_088,Susan Martinez,F,1986-10-15,780-24-9300,NaT,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Duplicate SSNs,NaN
1,app_016,Gary Wilson,M,1959-12-11,780-24-9300,NaT,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Duplicate SSNs,NaN
2,app_101,Sandra Smith,Female,1997-03-23,937-72-8731,NaT,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Duplicate SSNs,NaN
3,app_234,Samuel Hill,Male,1976/01/29,937-72-8731,NaT,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Duplicate SSNs,NaN
4,app_290,Stephanie Perez,Female,1990-01-06 00:00:00,866-48-3726,NaT,stephanie.perez69@gmail.com,172.25.130.219,90221,104000.0,39.0,0.27,-5000.0,True,<NA>,<NA>,4.3,49000.0,NaN,<NA>,0.0,0.0,0.0,0.0,0.0,599.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,Negative savings balance,NaN
5,app_190,Heather White,Female,1999-10-05 00:00:00,217-44-8191,2024-01-15 00:00:00+00:00,heather.white42@icloud.com,192.168.242.9,90297,0.0,0.0,0.14,19603.0,False,algorithm_risk_score,<NA>,NaN,NaN,NaN,<NA>,0.0,0.0,317.0,0.0,0.0,849.0,0.0,0.0,0.0,0.0,0.0,223.0,0.0,0.0,0.0,False,Zero income but positive spending,NaN
6,app_049,Donna Gonzalez,Female,2000-05-22 00:00:00,845-50-3562,NaT,donna.gonzalez12@yahoo.com,192.168.14.85,90278,59000.0,92.0,0.36,31881.0,False,algorithm_risk_score,<NA>,NaN,NaN,NaN,<NA>,0.0,0.0,0.0,0.0,848.0,444.0,0.0,131.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,Credit history exceeds legal possible limit,25.0
7,app_043,Daniel King,Male,1985-08-17 00:00:00,166-23-2000,NaT,daniel.king27@mail.com,10.204.248.60,10038,131000.0,-10.0,0.06,53098.0,True,<NA>,<NA>,3.9,68000.0,NaN,<NA>,0.0,0.0,0.0,0.0,464.0,0.0,0.0,398.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,Invalid credit history (negative),40.0
8,app_156,Jessica Green,Female,1999-11-28 00:00:00,801-19-4595,2024-01-15 00:00:00+00:00,jessica.green86@gmail.com,192.168.59.201,90239,25000.0,-3.0,0.21,13641.0,False,algorithm_risk_score,<NA>,NaN,NaN,NaN,<NA>,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,218.0,0.0,0.0,0.0,0.0,False,Invalid credit history (negative),26.0
9,app_075,Margaret Williams,Unknown,NaT,<NA>,NaT,<NA>,<NA>,<NA>,61000.0,29.0,0.15,25894.0,True,<NA>,<NA>,5.1,22000.0,NaN,<NA>,0.0,0.0,0.0,0.0,882.0,0.0,0.0,0.0,0.0,0.0,0.0,121.0,59.0,0.0,0.0,NaN,Missing either age or annual income,NaN


### In the beggining of the data cleaning there were 502 rows, in the cleaned dataset (df_clean) there are 482 rows. 18 of those records went into the df_flagged datase, 2 of them weren´t because the notes columns in the original dataset gave the reason for why they were invalid

In [ ]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 482 entries, 0 to 481
Data columns (total 35 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   application_id             482 non-null    object             
 1   processing_timestamp       59 non-null     datetime64[ns, UTC]
 2   full_name                  482 non-null    string             
 3   email                      479 non-null    string             
 4   ssn                        481 non-null    string             
 5   ip_address                 481 non-null    string             
 6   gender                     482 non-null    object             
 7   date_of_birth              482 non-null    datetime64[ns]     
 8   zip_code                   482 non-null    string             
 9   annual_income              482 non-null    float64            
 10  credit_history_months      482 non-null    int64              
 11  debt_t